# RAG Demo — Pipeline Walkthrough

A minimal, modular Retrieval-Augmented Generation pipeline. Each stage is a standalone module that reads from one folder under `data/` and writes to the next, so you can run, swap, or upgrade each block independently.

**Stages** (built incrementally):
1. **Document processing** — load raw files, extract to markdown ← *covered below*
2. **Chunker** — split markdown into overlapping chunks ← *covered below*
3. **Embedder** — turn chunks into vectors (local or OpenAI) ← *covered below*
4. **Retriever** — top-k chunks for a query ← *covered below*
5. Answerer — Claude generates a cited answer

## Stage 1 — Document Processing

- **Input:** `data/raw/` — drop your PDFs, `.md`, or `.txt` files here.
- **Output:** `data/markdown/` — one `.md` per input file, with YAML frontmatter (`title`, `authors`, `source`).
- **Extractor:** `pymupdf` for both PDF text and metadata (see `document_processing.pdf_proc.extract_pdf_basic`); pass-through for native text/markdown.

Two entry points:
- `extract_text(file_path)` — pure function, takes one file, returns a `Document`. Use it to inspect the result for a single file.
- `extract_docs(input_dir, output_dir)` — runs `extract_text` over a folder and writes `.md` (with frontmatter) to disk.

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from document_processing import extract_text, extract_docs
from chunker import chunk_doc, chunk_docs
from embedder import LocalEmbedder, ApiEmbedder, ParquetStore, embed_chunk, embed_chunks
from retriever import retrieve

In [2]:
RAW_DIR = ROOT_DIR / "data" / "raw"
MARKDOWN_DIR = ROOT_DIR / "data" / "markdown"
CHUNKS_DIR = ROOT_DIR / "data" / "chunks"
EMBEDDINGS_DIR = ROOT_DIR / "data" / "embeddings"
RAW_DIR.mkdir(parents=True, exist_ok=True)
MARKDOWN_DIR.mkdir(parents=True, exist_ok=True)
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

print("raw dir:       ", RAW_DIR)
print("markdown dir:  ", MARKDOWN_DIR)
print("chunks dir:    ", CHUNKS_DIR)
print("embeddings dir:", EMBEDDINGS_DIR)

raw dir:        /home/amir/source/rag-demo/data/raw
markdown dir:   /home/amir/source/rag-demo/data/markdown
chunks dir:     /home/amir/source/rag-demo/data/chunks
embeddings dir: /home/amir/source/rag-demo/data/embeddings


### What's in `data/raw/`?

If empty, drop a PDF (or `.md`/`.txt`) in there and re-run.

In [3]:
sorted(path.name for path in RAW_DIR.iterdir() if path.is_file())

['Large Language Models the New Scrum Masters.pdf']

### Single-file extraction (pure function)

Run `extract_text()` on one file and inspect the resulting `Document`.

In [4]:
input_files = [path for path in sorted(RAW_DIR.iterdir()) if path.is_file()]
assert input_files, f"No files in {RAW_DIR}. Drop a PDF/.md/.txt and re-run."

sample_file = input_files[0]
doc = extract_text(file_path=sample_file)

print("source:  ", doc.source.name)
print("metadata:", doc.metadata)
print("length:  ", len(doc.text), "chars")
print()
print("--- first 1500 chars ---")
print(doc.text[:1500])

source:   Large Language Models the New Scrum Masters.pdf
metadata: {'title': 'Large Language Models, the New Scrum Masters', 'authors': ['Juan Couder and Omar Ochoa']}
length:   23223 chars

--- first 1500 chars ---
Papers 
RED Innovation: Using Scrum to Develop an 
Agile Department 
2024 
Large Language Models, the New Scrum Masters 
Large Language Models, the New Scrum Masters 
Juan Couder 
ortizcoj@my.erau.edu 
Omar Ochoa 
ochoao@erau.edu 
Follow this and additional works at: https://commons.erau.edu/red-papers 
Scholarly Commons Citation 
Scholarly Commons Citation 
Couder, J., & Ochoa, O. (2024). Large Language Models, the New Scrum Masters. , (). Retrieved from 
https://commons.erau.edu/red-papers/2 
This Paper is brought to you for free and open access by the RED Innovation: Using Scrum to Develop an Agile 
Department at Scholarly Commons. It has been accepted for inclusion in Papers by an authorized administrator of 
Scholarly Commons. For more information, please contact comm

### Batch run (stage runner)

`extract_docs` extracts every supported file in `data/raw/` and writes a corresponding `.md` (with YAML frontmatter) to `data/markdown/`.

In [5]:
docs = extract_docs(input_dir=RAW_DIR, output_dir=MARKDOWN_DIR)
for doc in docs:
    print(f"{doc.source.name:40s} -> {len(doc.text):>8} chars  {doc.metadata}")

Large Language Models the New Scrum Masters.pdf ->    23223 chars  {'title': 'Large Language Models, the New Scrum Masters', 'authors': ['Juan Couder and Omar Ochoa']}


### Inspect the output folder

In [6]:
for path in sorted(MARKDOWN_DIR.iterdir()):
    if path.is_file() and path.suffix == ".md":
        print(f"{path.name:40s} {path.stat().st_size:>8} bytes")

Large Language Models the New Scrum Masters.md    23424 bytes
Large Language Models the New Scrum Masters_v1.md    23865 bytes


## Stage 2 — Chunker

- **Input:** `data/markdown/` — `.md` files with YAML frontmatter from stage 1.
- **Output:** `data/chunks/` — one `.jsonl` per source doc, one chunk per line.
- **Strategy:** fixed-size character windows with overlap (default `size=2000`, `overlap=200`).

Two entry points:
- `chunk_doc(doc)` — pure function over an in-memory `Document`.
- `chunk_docs(input_dir, output_dir)` — reads `.md` files via frontmatter, chunks, writes `.jsonl`.

In [7]:
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 200

all_chunks = [chunk for doc in docs for chunk in chunk_doc(doc=doc, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)]
print(f"{len(all_chunks)} chunks across {len(docs)} docs (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})\n")

first_chunk = all_chunks[0]
print("first chunk:")
print(f"  id:       {first_chunk.id}")
print(f"  source:   {first_chunk.source}")
print(f"  index:    {first_chunk.index}")
print(f"  metadata: {first_chunk.metadata}")
print(f"  text[:300]:\n{first_chunk.text[:300]}")

13 chunks across 1 docs (size=2000, overlap=200)

first chunk:
  id:       Large Language Models the New Scrum Masters#0
  source:   Large Language Models the New Scrum Masters.pdf
  index:    0
  metadata: {'title': 'Large Language Models, the New Scrum Masters', 'authors': ['Juan Couder and Omar Ochoa']}
  text[:300]:
Papers 
RED Innovation: Using Scrum to Develop an 
Agile Department 
2024 
Large Language Models, the New Scrum Masters 
Large Language Models, the New Scrum Masters 
Juan Couder 
ortizcoj@my.erau.edu 
Omar Ochoa 
ochoao@erau.edu 
Follow this and additional works at: https://commons.erau.edu/red-pap


### Batch run

`chunk_docs` reads every `.md` in `data/markdown/` (frontmatter + body), chunks each, and writes one `.jsonl` per doc to `data/chunks/`.

In [8]:
all_chunks_written = chunk_docs(input_dir=MARKDOWN_DIR, output_dir=CHUNKS_DIR, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP)
print(f"wrote {len(all_chunks_written)} chunks total (size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP})\n")

for path in sorted(CHUNKS_DIR.iterdir()):
    if path.suffix == ".jsonl":
        print(f"{path.name:40s} {path.stat().st_size:>8} bytes")

wrote 27 chunks total (size=2000, overlap=200)

Large Language Models the New Scrum Masters.jsonl    29573 bytes
Large Language Models the New Scrum Masters_v1.jsonl    29466 bytes


## Stage 3 — Embedder

- **Input:** `data/chunks/` — `.jsonl` files from stage 2.
- **Output:** whatever the chosen `data_store` writes. The default `ParquetStore` writes one `.parquet` per source doc to `data/embeddings/` with columns `chunk_id, source, text, vector, model, dim`.
- **Embedder backends** (swap with one line):
  - `LocalEmbedder()` — `sentence-transformers/all-MiniLM-L6-v2`, 384-dim, CPU, no API key.
  - `ApiEmbedder()` — OpenAI `text-embedding-3-small`, 1536-dim. Reads `OPENAI_API_KEY` from a `.env` file at the repo root (or accepts an explicit `api_key=` arg).
- **Storage backends:** any class in `embedder/data_stores.py` exposing `add(embeddings, source_stem)` and `load()`. `ParquetStore` is the only one for now; FAISS / Chroma can be added as sibling classes in the same file.

Two entry points (mirror earlier stages):
- `embed_chunk(chunk, embedder)` — pure function, embeds one chunk.
- `embed_chunks(input_dir, embedder, data_store)` — batches the calls per source file, hands each batch to `data_store.add(...)`.

> To use the API backend, create a `.env` at the repo root with `OPENAI_API_KEY=sk-...`. `.env` is gitignored.

In [9]:
# default to local: zero-friction, runs on CPU, no API key required
embedder = LocalEmbedder()
# swap to OpenAI by uncommenting the next line (requires .env with OPENAI_API_KEY)
# embedder = ApiEmbedder()

print(f"backend: {type(embedder).__name__}  model: {embedder.model}  dim: {embedder.dimension}")

/home/amir/miniconda3/envs/llm/lib/python3.12/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


backend: LocalEmbedder  model: sentence-transformers/all-MiniLM-L6-v2  dim: 384


### Single-chunk embedding (pure function)

In [10]:
sample_chunk = all_chunks[0]
embedding = embed_chunk(chunk=sample_chunk, embedder=embedder)

print(f"chunk_id:   {embedding.chunk_id}")
print(f"source:     {embedding.source}")
print(f"model:      {embedding.model}")
print(f"dim:        {embedding.dim}")
print(f"vector[:8]: {embedding.vector[:8]}")

chunk_id:   Large Language Models the New Scrum Masters#0
source:     Large Language Models the New Scrum Masters.pdf
model:      sentence-transformers/all-MiniLM-L6-v2
dim:        384
vector[:8]: [-0.03548708185553551, -0.07033353298902512, -0.03751732409000397, -0.018446998670697212, -0.0030614244751632214, 0.041072964668273926, -0.12445371598005295, 0.05954372510313988]


### Batch run

`embed_chunks` reads every `.jsonl` in `data/chunks/`, embeds each in batches, and writes one `.parquet` per source doc to `data/embeddings/`.

In [11]:
data_store = ParquetStore(directory=EMBEDDINGS_DIR)
all_embeddings = embed_chunks(input_dir=CHUNKS_DIR, embedder=embedder, data_store=data_store)
print(f"wrote {len(all_embeddings)} embeddings (model={embedder.model}, dim={embedder.dimension})\n")

for path in sorted(EMBEDDINGS_DIR.iterdir()):
    if path.suffix == ".parquet":
        print(f"{path.name:40s} {path.stat().st_size:>8} bytes")

wrote 27 embeddings (model=sentence-transformers/all-MiniLM-L6-v2, dim=384)

Large Language Models the New Scrum Masters.parquet    63406 bytes
Large Language Models the New Scrum Masters_v1.parquet    66439 bytes


### Working with the parquet output

Three patterns the format makes cheap: inspect the schema (parquet is self-describing), read only a subset of columns (no need to pull `text`/`vector` if you don't want them), and load a full row back as a dict.

In [12]:
import pyarrow.parquet as pq

sample_parquet = next(path for path in sorted(EMBEDDINGS_DIR.iterdir()) if path.suffix == ".parquet")

# 1) schema inspection — types and columns live in the file's footer
parquet_file = pq.ParquetFile(source=sample_parquet)
print(f"file:    {sample_parquet.name}")
print(f"rows:    {parquet_file.metadata.num_rows}")
print(f"schema:\n{parquet_file.schema_arrow}\n")

# 2) column projection — pull only the cheap columns, skip text + vector
metadata_columns = pq.read_table(source=sample_parquet, columns=["chunk_id", "source", "model", "dim"])
print("light read (chunk_id, source, model, dim) — first 3 rows:")
print(metadata_columns.slice(offset=0, length=3))

# 3) full row — load one complete record back as a dict
first_row = pq.read_table(source=sample_parquet).slice(offset=0, length=1).to_pylist()[0]
print(f"\nfirst row keys:    {list(first_row.keys())}")
print(f"first vector[:5]:  {first_row['vector'][:5]}")
print(f"first text[:120]:  {first_row['text'][:120]}")

file:    Large Language Models the New Scrum Masters.parquet
rows:    13
schema:
chunk_id: string
source: string
text: string
vector: list<element: double>
  child 0, element: double
model: string
dim: int64

light read (chunk_id, source, model, dim) — first 3 rows:
pyarrow.Table
chunk_id: string
source: string
model: string
dim: int64
----
chunk_id: [["Large Language Models the New Scrum Masters#0","Large Language Models the New Scrum Masters#1","Large Language Models the New Scrum Masters#2"]]
source: [["Large Language Models the New Scrum Masters.md","Large Language Models the New Scrum Masters.md","Large Language Models the New Scrum Masters.md"]]
model: [["sentence-transformers/all-MiniLM-L6-v2","sentence-transformers/all-MiniLM-L6-v2","sentence-transformers/all-MiniLM-L6-v2"]]
dim: [[384,384,384]]

first row keys:    ['chunk_id', 'source', 'text', 'vector', 'model', 'dim']
first vector[:5]:  [-0.0354870967566967, -0.07033351063728333, -0.03751732409000397, -0.018446998670697212, 

## Stage 4 — Retriever

- **Input:** the same `data_store` produced by stage 3 (any class with `load() -> list[Embedding]`).
- **Output:** in-memory `list[RetrievalResult]` returned to the caller — no on-disk artifact, since results are query-time.
- **Strategy:** embed the query with the *same* embedder used at index time, score every stored vector by cosine similarity (numpy matmul), return the top-k.

One entry point:
- `retrieve(query, embedder, data_store, top_k)` — returns `RetrievalResult(chunk_id, source, text, score)` for the best matches.

> The embedder must match the one used to populate the store. Mixing models silently degrades quality — that's why each parquet row records `model` and `dim`.

In [ ]:
query = "How can large language models support a Scrum Master?"
results = retrieve(query=query, embedder=embedder, data_store=data_store, top_k=3)

print(f"query: {query}\n")
for rank, result in enumerate(results, start=1):
    print(f"#{rank}  score={result.score:.4f}  {result.chunk_id}  ({result.source})")
    print(f"    {result.text[:240].replace(chr(10), ' ')}...")
    print()